In [1]:
DOCS = [
    {"id": "1", "title": "Remote Work Policy", "category": "HR",
     "text": "Contoso employees hired after January 2023 may work remotely up to "
             "three days per week. Fully remote arrangements require director approval "
             "and a signed home-office agreement. Equipment stipends of up to $500 are "
             "available annually for remote staff."},
    {"id": "2", "title": "Paid Time Off", "category": "HR",
     "text": "Full-time staff accrue 15 vacation days in their first year and 20 days "
             "after three years of service. Unused vacation up to 5 days may carry over "
             "into the next calendar year. Sick leave is separate and capped at 10 days."},
    {"id": "3", "title": "Expense Reimbursement", "category": "Finance",
     "text": "Business expenses must be submitted within 30 days with itemized receipts. "
             "Meals are reimbursed up to $75 per day during travel. Airfare must be booked "
             "in economy class unless the flight exceeds six hours."},
    {"id": "4", "title": "Security Onboarding", "category": "IT",
     "text": "All new hires must complete multi-factor authentication setup and annual "
             "security awareness training within their first week. Company laptops are "
             "encrypted at rest and managed through Intune."},
]


In [2]:
def chunk_text(text, max_chars=1200, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + max_chars])
        start += max_chars - overlap
    return chunks

In [3]:
from openai import OpenAI

models = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # key required but ignored
EMBED_MODEL = "nomic-embed-text"   # 768-dimensional

def embed(texts):
    resp = models.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]


In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch,
)

SEARCH_ENDPOINT = "https://srch-rag-29276.search.windows.net/"
# ADMIN_KEY = ""
INDEX_NAME = "contoso-kb"

cred = DefaultAzureCredential()   # after `az login`
index_client = SearchIndexClient(SEARCH_ENDPOINT, credential=cred)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="title", type=SearchFieldDataType.String),
    SearchableField(name="chunk", type=SearchFieldDataType.String),
    SimpleField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=768,          # MUST match nomic-embed-text
        vector_search_profile_name="vprofile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

semantic = SemanticSearch(configurations=[
    SemanticConfiguration(
        name="semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="chunk")],
        ),
    )
])

index_client.create_or_update_index(
    SearchIndex(name=INDEX_NAME, fields=fields,
                vector_search=vector_search, semantic_search=semantic)
)
print(f"Index '{INDEX_NAME}' created.")


Index 'contoso-kb' created.


In [5]:
from azure.search.documents import SearchClient

# search_client = SearchClient(SEARCH_ENDPOINT, INDEX_NAME, AzureKeyCredential(ADMIN_KEY))

search_client = SearchClient(SEARCH_ENDPOINT, INDEX_NAME, cred)

payload = []
for doc in DOCS:
    for i, ch in enumerate(chunk_text(doc["text"])):
        payload.append({
            "id": f'{doc["id"]}-{i}',
            "title": doc["title"],
            "chunk": ch,
            "category": doc["category"],
            "contentVector": embed([ch])[0],
        })

result = search_client.upload_documents(documents=payload)
print(f"Uploaded {len(payload)} chunks. First status: {result[0].succeeded}")


Uploaded 4 chunks. First status: True


In [6]:
q = "How many vacation days do I get after three years?"
for r in search_client.search(search_text=q, select=["title","chunk"], top=3):
    print(round(r["@search.score"], 3), r["title"])

2.7 Paid Time Off
0.846 Remote Work Policy
0.177 Expense Reimbursement


In [7]:
from azure.search.documents.models import VectorizedQuery

vq = VectorizedQuery(vector=embed([q])[0], k_nearest_neighbors=3, fields="contentVector")
for r in search_client.search(search_text=None, vector_queries=[vq],
                              select=["title","chunk"], top=3):
    print(round(r["@search.score"], 3), r["title"])

0.807 Paid Time Off
0.685 Remote Work Policy
0.68 Expense Reimbursement


In [8]:
vq = VectorizedQuery(vector=embed([q])[0], k_nearest_neighbors=5, fields="contentVector")
results = search_client.search(
    search_text=q,                        # keyword arm
    vector_queries=[vq],                  # vector arm
    query_type="semantic",                # cross-encoder re-rank
    semantic_configuration_name="semantic-config",
    select=["title","chunk","category"],
    top=5,
)
for r in results:
    print(round(r["@search.reranker_score"], 2), r["title"])

3.07 Paid Time Off
1.59 Expense Reimbursement
1.23 Remote Work Policy
1.0 Security Onboarding


In [9]:
results = search_client.search(
    search_text="reimbursement rules", vector_queries=[vq],
    filter="category eq 'Finance'", top=3, select=["title","chunk"])

In [10]:
GROUNDED_PROMPT = """You are a helpful assistant for Contoso employees.

Use only the sources below to answer the question.

Rules:
- If the answer is stated in the sources, answer it directly.
- Cite the source title in square brackets, for example [Paid Time Off].
- Reply exactly "I don't have that information." only when none of the sources contains the answer.

Question:
{query}

Sources:
{sources}

Answer:
"""

In [11]:
CHAT_MODEL = "llama3.2:3b"

def rag_answer(query):
    vq = VectorizedQuery(vector=embed([query])[0], k_nearest_neighbors=5, fields="contentVector")
    hits = search_client.search(
        search_text=query, vector_queries=[vq],
        query_type="semantic", semantic_configuration_name="semantic-config",
        select=["title","chunk"], top=5,
    )
    sources = "\n=====\n".join(f'[{h["title"]}] {h["chunk"]}' for h in hits)

    resp = models.chat.completions.create(         # same in-cluster client as embeddings
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "user",
                   "content": GROUNDED_PROMPT.format(query=query, sources=sources)}],
    )
    return resp.choices[0].message.content

print(rag_answer("Can I carry over unused vacation days?"))
print(rag_answer("What is Contoso's parental leave policy?"))   # expect the refusal string

According to [Paid Time Off], unused vacation days up to 5 days may carry over into the next calendar year.
I don't have that information.
